In [2]:
!pip install pyyaml

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip available: 22.3 -> 25.2
[notice] To update, run: C:\Program Files\Python311\python.exe -m pip install --upgrade pip


In [22]:
def add_metadata_to_table(table_name: str, fields_description: dict, tbl_description: str = None, tag_dict: dict = None, fields_types: dict = None):
    """
    Adiciona comentários e tags a uma tabela e suas colunas no Databricks.
    Também verifica se o tipo da coluna no YAML bate com o tipo real do Databricks.
    """

    # 1. Verifica colunas existentes
    try:
        df = spark.table(table_name)
        table_columns = {field.name for field in df.schema.fields}
        table_types = {field.name: field.dataType.simpleString() for field in df.schema.fields}
    except Exception as e:
        print(f"[ERRO] 🔴 Falha ao acessar a tabela '{table_name}': {e}")
        return

    # 2. Adiciona comentários nas colunas
    for column, metadata in fields_description.items():
        if column in table_columns:
            # Verifica o tipo se informado no YAML
            expected_type = None
            if fields_types and column in fields_types:
                expected_type = fields_types[column]

                real_type = table_types[column]
                if real_type.lower() != expected_type.lower():
                    print(f"[WARN] 🟡 Coluna '{column}' com tipo diferente: YAML='{expected_type}' | Databricks='{real_type}'")

            # Cria comentário
            comment_parts = [f"{key}: {value}" for key, value in metadata.items()]
            comment_str = " | ".join(comment_parts).replace("'", "''")
            alter_stmt = f"COMMENT ON COLUMN {table_name}.`{column}` IS '{comment_str}'"
            try:
                spark.sql(alter_stmt)
                print(f"[OK] 🟢 Comentário adicionado à coluna '{column}'")
            except Exception as e:
                print(alter_stmt)
                print(f"[ERRO] 🔴 Falha ao comentar coluna '{column}': {e}")
        else:
            print(f"[SKIP] 🟡 Coluna '{column}' não existe na tabela '{table_name}'")
            
            
    # 3. Adiciona descrição à tabela
    if tbl_description:
        description_clean = tbl_description.replace("'", "''")
        try:
            spark.sql(f"COMMENT ON TABLE {table_name} IS '{description_clean}'")
            print(f"[OK] 🟢 Comentário adicionado à tabela '{table_name}'")
        except Exception as e:
            print(f"[ERRO] 🔴 Falha ao comentar a tabela '{table_name}': {e}")

    # 4. Adiciona tags à tabela (usando ALTER TABLE SET TBLPROPERTIES)
    # 4. Tags da tabela
    if tag_dict:
        tag_parts = []
        for k, v in tag_dict.items():
            v_escaped = v.replace("'", "''")  # escapa aspas simples
            tag_parts.append(f"'{k}' = '{v_escaped}'")
        tag_str = ", ".join(tag_parts)
        try:
            spark.sql(f"ALTER TABLE {table_name} SET TBLPROPERTIES ({tag_str})") 
            print(f"[OK] 🟢 Tags adicionadas à tabela '{table_name}'")
        except Exception as e:
            print(f"[ERRO] 🔴 Falha ao adicionar tags à tabela '{table_name}': {e}")



In [18]:
import os
from string import Template
import yaml

# Variável que será usada na substituição
bundle_target = "prd"

# Pasta onde estão os arquivos YAML
yaml_folder = "."

# Lista todos os arquivos da pasta
for filename in os.listdir(yaml_folder):
    if filename.endswith(".yml") or filename.endswith(".yaml"):
        filepath = os.path.join(yaml_folder, filename)
        
        # Lê o YAML como texto
        with open(filepath, "r", encoding="utf-8") as f:
            yaml_text = f.read()
        
        # Substitui variáveis
        yaml_text = Template(yaml_text).safe_substitute({"bundle.target": bundle_target})
        
        # Carrega o YAML
        dados = yaml.safe_load(yaml_text)
        tabelas = dados.get("tables", {})
        
        print(f"\n===== Arquivo: {filename} =====")
        
        for tabela_nome, tabela_info in tabelas.items():
            print(f"\nNome da tabela: {tabela_info.get('table_path', 'Sem Table Path')}")

            tabela_metadata = tabela_info.get("metadata")
            print(f"   Metadata: {tabela_metadata if tabela_metadata else 'Nenhum metadata'}")

            colunas = tabela_info.get("columns", [])
            if colunas:
                print("   Colunas:")
                for coluna in colunas:
                    nome_coluna = coluna.get("name", "sem_nome")
                    tipo_coluna = coluna.get("type", "sem tipo")

                    if "type" in coluna:
                        print(f"     - {nome_coluna} ({tipo_coluna}) certo")
                    else:
                        print(f"     - {nome_coluna} ({tipo_coluna})")

                    coluna_metadata = coluna.get("metadata")
                    print(f"         metadata: {coluna_metadata if coluna_metadata else '{}'}")
            else:
                print("   Nenhuma coluna")

            print("\n" + "="*40)



===== Arquivo: tabela1 copy.yml =====

Nome da tabela: _estruturante.silver_postgresql_argilla.teste
   Metadata: {'product': 'TACIA', 'data_owner': 'deyvid.lima@yduqs.com.br', 'data_tech_owner': 'andre.russi@yduqs.com.br;deyvid.lima@yduqs.com.br', 'data_source_repository': 'https://arquiteturaestacio.visualstudio.com/YDUQS%20-%20AI/_git/ia-yduqs-tacia-metadados', 'nivel_compartilhamento': 'publico'}
   Colunas:
     - coluna1 (string) certo
         metadata: {'descricao': 'Classificação da pergunta', 'categoria': 'Classificação', 'detalhamento': 'Classificação atribuída à pergunta', 'origem': '${bundle.target}_estruturante.silver_postgresql_argilla.responses'}
     - coluna2 (string) certo
         metadata: {'descricao': 'Coluna 1 é a chave primária', 'categoria': 'Classificação 2', 'tipo': 'string'}
     - coluna3 (string) certo
         metadata: {'descricao': 'Coluna 1 é a chave primária', 'categoria': 'Classificação 3', 'categoria3': 'Classificação 4', 'tipo': 'string'}


=====

In [ ]:
import os
from string import Template
import yaml

# Variável que será usada na substituição
bundle_target = "prd"
yaml_folder = "."

for filename in os.listdir(yaml_folder):
    if filename.endswith(".yml") or filename.endswith(".yaml"):
        filepath = os.path.join(yaml_folder, filename)
        
        with open(filepath, "r", encoding="utf-8") as f:
            yaml_text = f.read()
        
        # substitui variáveis do YAML (${bundle.target})
        yaml_text = Template(yaml_text).safe_substitute({"bundle.target": bundle_target})
        
        dados = yaml.safe_load(yaml_text)
        tabelas = dados.get("tables", {})

        print(f"\n===== Arquivo: {filename} =====")

        for tabela_nome, tabela_info in tabelas.items():
            table_path = tabela_info.get("table_path")
            if not table_path:
                print("[SKIP] Nenhum table_path definido.")
                continue

            tabela_metadata = tabela_info.get("metadata", {})
            tbl_description = tabela_metadata.get("description")
            tag_dict = tabela_metadata.get("tags", {})

            # monta dict de colunas -> metadados
            colunas = tabela_info.get("columns", [])
            fields_description = {col.get("name", "sem_nome"): col.get("metadata", {}) for col in colunas}
            fields_types = {col.get("name", "sem_nome"): col.get("type", "").lower() for col in colunas}

            add_metadata_to_table(
                table_name=table_path,
                fields_description=fields_description,
                tbl_description=tbl_description,
                tag_dict=tag_dict,
                fields_types=fields_types
            )


TypeError: string indices must be integers

In [ ]:
def add_metadata_to_table(table_name: str, fields_description: dict, tbl_description: str = None, tag_dict: dict = None, fields_types: dict = None):
    """
    Adiciona comentários e tags a uma tabela e suas colunas no Databricks.
    Também verifica se o tipo da coluna no YAML bate com o tipo real do Databricks.
    """

    # 1. Verifica colunas existentes
    try:
        df = spark.table(table_name)
        table_columns = {field.name for field in df.schema.fields}
        table_types = {field.name: field.dataType.simpleString() for field in df.schema.fields}
    except Exception as e:
        print(f"[ERRO] 🔴 Falha ao acessar a tabela '{table_name}': {e}")
        return

    # 2. Adiciona comentários nas colunas
    for column, metadata in fields_description.items():
        if column in table_columns:
            # Verifica o tipo se informado no YAML
            expected_type = None
            if fields_types and column in fields_types:
                expected_type = fields_types[column]

                real_type = table_types[column]
                if real_type.lower() != expected_type.lower():
                    print(f"[WARN] 🟡 Coluna '{column}' com tipo diferente: YAML='{expected_type}' | Databricks='{real_type}'")

            # Cria comentário
            comment_parts = [f"{key}: {value}" for key, value in metadata.items()]
            comment_str = " | ".join(comment_parts).replace("'", "''")
            alter_stmt = f"COMMENT ON COLUMN {table_name}.`{column}` IS '{comment_str}'"
            try:
                spark.sql(alter_stmt)
                print(f"[OK] 🟢 Comentário adicionado à coluna '{column}'")
            except Exception as e:
                print(alter_stmt)
                print(f"[ERRO] 🔴 Falha ao comentar coluna '{column}': {e}")
        else:
            print(f"[SKIP] 🟡 Coluna '{column}' não existe na tabela '{table_name}'")
            
            
    # 3. Adiciona descrição à tabela
    if tbl_description:
        description_clean = tbl_description.replace("'", "''")
        try:
            spark.sql(f"COMMENT ON TABLE {table_name} IS '{description_clean}'")
            print(f"[OK] 🟢 Comentário adicionado à tabela '{table_name}'")
        except Exception as e:
            print(f"[ERRO] 🔴 Falha ao comentar a tabela '{table_name}': {e}")

    # 4. Adiciona tags à tabela (usando ALTER TABLE SET TBLPROPERTIES)
    # 4. Tags da tabela
    if tag_dict:
        tag_parts = []
        for k, v in tag_dict.items():
            v_escaped = v.replace("'", "''")  # escapa aspas simples
            tag_parts.append(f"'{k}' = '{v_escaped}'")
        tag_str = ", ".join(tag_parts)
        try:
            spark.sql(f"ALTER TABLE {table_name} SET TBLPROPERTIES ({tag_str})") 
            print(f"[OK] 🟢 Tags adicionadas à tabela '{table_name}'")
        except Exception as e:
            print(f"[ERRO] 🔴 Falha ao adicionar tags à tabela '{table_name}': {e}")

import os
from string import Template
import yaml

# Variável que será usada na substituição
bundle_target = "prd"
yaml_folder = "."

for filename in os.listdir(yaml_folder):
    if filename.endswith(".yml") or filename.endswith(".yaml"):
        filepath = os.path.join(yaml_folder, filename)
        
        with open(filepath, "r", encoding="utf-8") as f:
            yaml_text = f.read()
        
        # substitui variáveis do YAML (${bundle.target})
        yaml_text = Template(yaml_text).safe_substitute({"bundle.target": bundle_target})
        
        dados = yaml.safe_load(yaml_text)
        tabelas = dados.get("tables", {})

        print(f"\n===== Arquivo: {filename} =====")

        for tabela_nome, tabela_info in tabelas.items():
            table_path = tabela_info.get("table_path")
            if not table_path:
                print("[SKIP] Nenhum table_path definido.")
                continue

            tabela_metadata = tabela_info.get("metadata", {})
            tbl_description = tabela_metadata.get("description")
            tag_dict = tabela_metadata.get("tags", {})

            # monta dict de colunas -> metadados
            colunas = tabela_info.get("columns", [])
            fields_description = {col.get("name", "sem_nome"): col.get("metadata", {}) for col in colunas}
            fields_types = {col.get("name", "sem_nome"): col.get("type", "").lower() for col in colunas}

            add_metadata_to_table(
                table_name=table_path,
                fields_description=fields_description,
                tbl_description=tbl_description,
                tag_dict=tag_dict,
                fields_types=fields_types
            )


# Classes Mock para Testes

Abaixo estão as classes que simulam o comportamento do Spark para testes locais:
- `MockSpark`: Simula o objeto spark principal
- `MockDataFrame`: Simula um DataFrame do Spark
- `MockSchema`: Simula o schema de uma tabela
- `Field`: Representa um campo/coluna da tabela

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Any

@dataclass
class Field:
    name: str
    dataType: type

class DataType:
    def __init__(self, type_str: str = "string"):
        self._type = type_str
        
    def simpleString(self) -> str:
        return self._type

class MockSchema:
    def __init__(self, fields: List[Field]):
        self.fields = fields

class MockDataFrame:
    def __init__(self, schema: MockSchema):
        self.schema = schema

class MockSpark:
    def __init__(self, mock_tables: Dict[str, List[Dict[str, Any]]] = None):
        """
        Inicializa o MockSpark com tabelas simuladas
        
        Args:
            mock_tables: Dicionário com nome da tabela -> lista de dicionários com dados
        """
        self.mock_tables = mock_tables or {}
        
    def table(self, name: str) -> MockDataFrame:
        # Se a tabela existe no mock_tables, usa sua estrutura
        if name in self.mock_tables:
            table_data = self.mock_tables[name]
            if table_data:
                # Pega as colunas do primeiro registro
                columns = list(table_data[0].keys())
                fields = [Field(name=col, dataType=DataType()) for col in columns]
                return MockDataFrame(MockSchema(fields))
        
        # Caso contrário, retorna uma estrutura padrão
        fields = [
            Field(name="coluna1", dataType=DataType("string")),
            Field(name="coluna2", dataType=DataType("string")),
            Field(name="coluna3", dataType=DataType("string"))
        ]
        return MockDataFrame(MockSchema(fields))
    
    def sql(self, query: str) -> None:
        """Simula execução de queries SQL"""
        print(f"[MOCK] Executando query: {query}")

# Exemplo de dados mock para testes
MOCK_DATA = {
    "_estruturante.silver_postgresql_argilla.responses": [
        {"coluna1": "valor1", "coluna2": "valor2", "coluna3": "valor3"}
    ],
    "_estruturante.silver_postgresql_argilla.questions": [
        {"coluna1": "q1", "coluna2": "q2", "coluna3": "q3"}
    ]
}

# Funções Auxiliares

As funções abaixo ajudam no processamento dos arquivos YAML:
- `process_yaml_file`: Processa um arquivo YAML, substituindo variáveis
- `get_table_metadata`: Extrai os metadados de uma tabela do YAML

In [ ]:
import os
from string import Template
import yaml
from typing import Dict, Any, Tuple

def process_yaml_file(filepath: str, variables: Dict[str, str] = None) -> Dict[str, Any]:
    """
    Processa um arquivo YAML, substituindo variáveis
    
    Args:
        filepath: Caminho do arquivo YAML
        variables: Dicionário com variáveis para substituição
        
    Returns:
        Dados do YAML processado
    """
    with open(filepath, "r", encoding="utf-8") as f:
        yaml_text = f.read()
    
    # Substitui variáveis se fornecidas
    if variables:
        yaml_text = Template(yaml_text).safe_substitute(variables)
    
    return yaml.safe_load(yaml_text)

def get_table_metadata(table_info: Dict[str, Any]) -> Tuple[str, Dict[str, Any], Dict[str, Dict], Dict[str, str]]:
    """
    Extrai os metadados de uma tabela do YAML
    
    Args:
        table_info: Informações da tabela do YAML
        
    Returns:
        Tupla com:
        - table_path: Caminho da tabela
        - tag_dict: Dicionário com tags
        - fields_description: Dicionário com descrição dos campos
        - fields_types: Dicionário com tipos dos campos
    """
    table_path = table_info.get("table_path")
    
    # Extrai metadados da tabela (serão usados como tags)
    tabela_metadata = table_info.get("metadata", {})
    
    # Monta dicionários de metadados e tipos das colunas
    colunas = table_info.get("columns", [])
    fields_description = {
        col.get("name", "sem_nome"): col.get("metadata", {}) 
        for col in colunas
    }
    fields_types = {
        col.get("name", "sem_nome"): col.get("type", "").lower() 
        for col in colunas
    }
    
    return table_path, tabela_metadata, fields_description, fields_types

# Testes

Abaixo estão os testes do código:
1. Criação do ambiente de teste com dados mock
2. Processamento do arquivo YAML
3. Execução da função principal com os dados processados

In [ ]:
# Cria instância do MockSpark com dados de teste
spark = MockSpark(MOCK_DATA)

# Processa o arquivo YAML
yaml_data = process_yaml_file(
    "tabela1.yml",
    variables={"bundle.target": "prd"}
)

# Processa cada tabela do YAML
for tabela_nome, tabela_info in yaml_data.get("tables", {}).items():
    print(f"\n===== Processando tabela: {tabela_nome} =====")
    
    # Extrai metadados da tabela
    table_path, tag_dict, fields_description, fields_types = get_table_metadata(tabela_info)
    
    if not table_path:
        print("[SKIP] Nenhum table_path definido.")
        continue
    
    # Testa a função principal
    add_metadata_to_table(
        table_name=table_path,
        fields_description=fields_description,
        tbl_description=None,  # Não usado no YAML atual
        tag_dict=tag_dict,
        fields_types=fields_types
    )